# **Files and Exceptions**

**DS112 — Computer Programming**

**Week 8:** · **Suggested class time: 5 hours (2-hour lecture + 3-hour laboratory)**

**ILO :** Develop programs that read and write files and handle errors using exceptions.

Variables disappear when a program ends. To **keep** data — or to process data someone else collected — programs read and write **files**. And because files can be missing, empty, or full of surprises, programs also need a way to **handle errors** gracefully: **exceptions**.

```{mermaid}
flowchart LR
    F["file on disk<br/>grades.csv"] -->|read| P["your program"]
    P -->|write| R["report.txt"]
    P -.->|"something goes wrong"| X["exception<br/>handled with try/except"]
```

## **Paths**

A **path** tells Python where a file is.

* **Relative path** — from the current folder: `data/grades.csv`
* **Absolute path** — from the drive root: `C:/Users/Ana/DS112/data/grades.csv`

Prefer relative paths so your notebook works on other computers. The `pathlib` module makes paths easy:

In [ ]:
from pathlib import Path

data_file = Path("data") / "grades.csv"
print(data_file, data_file.exists(), data_file.suffix, data_file.name)

## **Reading Text Files**

Open files with `open()` inside a **`with` block**, which closes the file automatically — even if an error happens.

| Mode | Meaning |
| ---- | ------- |
| `"r"` | read (default) |
| `"w"` | write — creates or **overwrites** |
| `"a"` | append to the end |

In [ ]:
with open("data/grades.csv", encoding="utf-8") as f:
    header_line = f.readline()          # one line
    first_rows = [next(f) for _ in range(3)]

print(header_line.strip())
for line in first_rows:
    print(line.strip().split(","))

Notice that everything read from a text file is a **string** — numbers must be converted with `float()` or `int()`.

## **Reading CSV Files**

**CSV** (comma-separated values) is the most common format for tabular data. The `csv` module handles quoting and commas inside fields correctly.

In [ ]:
import csv

with open("data/grades.csv", encoding="utf-8") as f:
    reader = csv.DictReader(f)            # each row becomes a dictionary
    rows = list(reader)

print(len(rows), "rows; columns:", list(rows[0].keys()))
print(rows[0])

Some cells in this dataset are **empty** (a missing midterm score). Converting an empty string to a number fails — a perfect case for exceptions.

## **Exceptions**

An **exception** is an error that happens **while the program runs**. Without handling, the program stops and shows a traceback.

| Exception | Typical cause |
| --------- | ------------- |
| `ValueError` | `float("")`, `int("abc")` |
| `TypeError` | `"5" + 5` |
| `KeyError`, `IndexError` | a missing key / position |
| `ZeroDivisionError` | dividing by zero |
| `FileNotFoundError` | opening a file that doesn't exist |

### **`try` and `except`**

> **Analogy.** `try`/`except` is like a **safety net under a trapeze**: you *try* the move; if you fall, the net *catches* you and the show continues.

In [ ]:
def to_number(text):
    try:
        return float(text)
    except ValueError:
        return None


midterms = [to_number(r["midterm"]) for r in rows]
valid = [m for m in midterms if m is not None]
print(f"{len(valid)} valid, {midterms.count(None)} missing, average {sum(valid) / len(valid):.2f}")

### **The full form: `else` and `finally`**

```python
try:
    ...            # code that might fail
except SomeError:
    ...            # runs only if that error happened
else:
    ...            # runs only if NO error happened
finally:
    ...            # ALWAYS runs (cleanup)
```

In [ ]:
def safe_divide(a, b):
    try:
        result = a / b
    except ZeroDivisionError:
        print("  cannot divide by zero")
        return None
    else:
        print("  division OK")
        return result
    finally:
        print("  (finished)")


print(safe_divide(10, 4))
print(safe_divide(10, 0))

In [ ]:
try:
    with open("data/missing_file.csv") as f:
        f.read()
except FileNotFoundError as error:
    print("Could not open the file:", error.filename)

### **Raising exceptions**

Your own functions can **raise** an exception when they receive invalid input:

In [ ]:
def compute_final(quiz, lab, exam):
    for score in (quiz, lab, exam):
        if not 0 <= score <= 100:
            raise ValueError(f"Score {score} is outside 0–100.")
    return 0.3 * quiz + 0.3 * lab + 0.4 * exam


try:
    compute_final(90, 105, 80)
except ValueError as error:
    print("Rejected:", error)

### **Validating user input**

Combining a loop with `try`/`except` gives a robust input routine:

```python
while True:
    try:
        score = float(input("Enter a score: "))
        if 0 <= score <= 100:
            break
        print("Score must be between 0 and 100.")
    except ValueError:
        print("Please type a number.")
```

## **Writing Files**

Writing uses mode `"w"` (overwrite) or `"a"` (append). The cell below writes a small report to a **temporary** folder so the book's files stay unchanged.

In [ ]:
import tempfile

out_dir = Path(tempfile.mkdtemp())
report = out_dir / "class_report.csv"

with open(report, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["name", "final", "remark"])
    for r in rows:
        try:
            final = 0.3 * float(r["quiz"]) + 0.3 * float(r["lab"]) + 0.4 * float(r["final"])
        except ValueError:
            writer.writerow([r["name"], "", "INCOMPLETE"])
            continue
        writer.writerow([r["name"], round(final, 2), "Passed" if final >= 75 else "Failed"])

with open(report, encoding="utf-8") as f:
    for line in list(f)[:6]:
        print(line.strip())

## **Good Practices**

* Always use `with open(...)` so files are closed.
* Specify `encoding="utf-8"` for text files.
* Catch **specific** exceptions (`except ValueError`), never a bare `except:` that hides every error.
* Keep `try` blocks **small** — only around the line that may fail.
* Validate data as early as possible and give **clear** error messages.

## **Summary**

| Concept | Key idea |
| ------- | -------- |
| path | where a file is; prefer relative paths and `pathlib` |
| `with open()` | read (`"r"`), write (`"w"`), append (`"a"`); closes automatically |
| `csv` module | `reader`, `DictReader`, `writer` for tabular data |
| exception | a runtime error; handle with `try`/`except` |
| `else` / `finally` | success-only code / cleanup that always runs |
| `raise` | signal invalid input from your own functions |

## **References**

Downey, A. B. (2024). *Think Python: How to think like a computer scientist* (3rd ed.). O'Reilly Media. https://allendowney.github.io/ThinkPython/

Liang, Y. D. (2013). *Introduction to programming using Python*. Pearson.

Maulion, M. R. (2022). *DS 112 computer programming* [Instructional material]. University of Science and Technology of Southern Philippines.

Python Software Foundation. (2026). *Built-in exceptions*. https://docs.python.org/3/library/exceptions.html

Python Software Foundation. (2026). *csv — CSV file reading and writing*. https://docs.python.org/3/library/csv.html

Python Software Foundation. (2026). *The Python tutorial: 7. Input and output*. https://docs.python.org/3/tutorial/inputoutput.html

Python Software Foundation. (2026). *The Python tutorial: 8. Errors and exceptions*. https://docs.python.org/3/tutorial/errors.html